# IMPORTS

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Team performance and transfers dataset

## Joining datasets and filtering for scope

In [2]:
df_clubs = pd.read_csv("../data/raw/transf_perf/clubs.csv")
df_comps = pd.read_csv("../data/raw/transf_perf/competitions.csv")
df_games = pd.read_csv("../data/raw/transf_perf/games.csv")
df_transfers = pd.read_csv("../data/raw/transf_perf/transfers.csv")

In [3]:
def build_team_performance(df_games: pd.DataFrame) -> pd.DataFrame:
    """
    Build a team-season performance table from match-level game data.

    Output:
        one row per club_id x season x competition_id
        with points, wins, draws, losses, goals_for, goals_against,
        goal_difference, and matches_played
    """

    # Keep only columns needed for performance aggregation
    games = df_games[
        [
            "game_id",
            "competition_id",
            "season",
            "home_club_id",
            "away_club_id",
            "home_club_name",
            "away_club_name",
            "home_club_goals",
            "away_club_goals",
        ]
    ].copy()

    # Build home-team view 
    home = games.rename(
        columns={
            "home_club_id": "club_id",
            "home_club_name": "club_name",
            "home_club_goals": "goals_for",
            "away_club_goals": "goals_against",
        }
    )[
        [
            "game_id",
            "competition_id",
            "season",
            "club_id",
            "club_name",
            "goals_for",
            "goals_against",
        ]
    ].copy()

    # Assign match result for home team
    home["win"] = (home["goals_for"] > home["goals_against"]).astype(int)
    home["draw"] = (home["goals_for"] == home["goals_against"]).astype(int)
    home["loss"] = (home["goals_for"] < home["goals_against"]).astype(int)
    home["points"] = home["win"] * 3 + home["draw"]

    # Build away-team view 
    away = games.rename(
        columns={
            "away_club_id": "club_id",
            "away_club_name": "club_name",
            "away_club_goals": "goals_for",
            "home_club_goals": "goals_against",
        }
    )[
        [
            "game_id",
            "competition_id",
            "season",
            "club_id",
            "club_name",
            "goals_for",
            "goals_against",
        ]
    ].copy()

    # Assign match result for away team
    away["win"] = (away["goals_for"] > away["goals_against"]).astype(int)
    away["draw"] = (away["goals_for"] == away["goals_against"]).astype(int)
    away["loss"] = (away["goals_for"] < away["goals_against"]).astype(int)
    away["points"] = away["win"] * 3 + away["draw"]

    # Stack home and away rows into one long team-match table 
    team_matches = pd.concat([home, away], ignore_index=True)

    # Aggregate to team-season level 
    team_perf = (
        team_matches.groupby(
            ["season", "competition_id", "club_id", "club_name"], as_index=False
        )
        .agg(
            matches_played=("game_id", "count"),
            wins=("win", "sum"),
            draws=("draw", "sum"),
            losses=("loss", "sum"),
            goals_for=("goals_for", "sum"),
            goals_against=("goals_against", "sum"),
            points=("points", "sum"),
        )
    )

    # Create derived metrics
    team_perf["goal_difference"] = (
        team_perf["goals_for"] - team_perf["goals_against"]
    )

    # Add league position within each season x competition 
    # Standard ranking logic:
    # points desc, goal difference desc, goals_for desc
    team_perf = team_perf.sort_values(
        by=["season", "competition_id", "points", "goal_difference", "goals_for"],
        ascending=[True, True, False, False, False],
    ).copy()

    team_perf["position"] = (
        team_perf.groupby(["season", "competition_id"])
        .cumcount()
        .add(1)
    )

    return team_perf.reset_index(drop=True)

In [4]:
df_team_perf = build_team_performance(df_games)
df_team_perf.head()

,season,competition_id,club_id,club_name,matches_played,wins,draws,losses,goals_for,goals_against,points,goal_difference,position
0,2012,BE1,58,Royal Sporting Club Anderlecht,30,20,7,3,69,27,67,42,1
1,2012,BE1,3508,Sportvereniging Zulte Waregem,30,19,6,5,49,29,63,20,2
2,2012,BE1,1184,Koninklijke Racing Club Genk,30,15,10,5,63,40,55,23,3
3,2012,BE1,2282,Club Brugge Koninklijke Voetbalvereniging,30,15,9,6,66,43,54,23,4
4,2012,BE1,498,KSC Lokeren (- 2020),30,14,9,7,53,38,51,15,5


In [5]:
df_team_perf = build_team_performance(df_games)
print(df_team_perf.shape)
print(sorted(df_team_perf["competition_id"].astype(str).unique()))

(20463, 13)
['BE1', 'BESC', 'CDR', 'CGB', 'CIT', 'CL', 'CLQ', 'DFB', 'DFL', 'DK1', 'DKP', 'ECLQ', 'EL', 'ELQ', 'ES1', 'FAC', 'FR1', 'FRCH', 'GB1', 'GBCS', 'GR1', 'GRP', 'IT1', 'KLUB', 'L1', 'NL1', 'NLP', 'NLSC', 'PO1', 'POCP', 'POSU', 'RU1', 'RUP', 'RUSS', 'SC1', 'SCI', 'SFA', 'SUC', 'TR1', 'UCOL', 'UKR1', 'UKRP', 'UKRS', 'USC']


## Only keep major 5 leagues

In [6]:
TOP5_IDS = ["GB1", "ES1", "IT1", "FR1", "L1"]

df_team_perf = df_team_perf[
    df_team_perf["competition_id"].astype(str).isin(TOP5_IDS)
].copy()

# ADD THIS — filter to scope years
df_team_perf = df_team_perf[
    (df_team_perf['season'] >= 2014) &
    (df_team_perf['season'] <= 2024)
].copy()

print(df_team_perf.shape)
print(df_team_perf["competition_id"].value_counts())
print("Seasons:", sorted(df_team_perf['season'].unique()))

(1074, 13)
competition_id
ES1    220
GB1    220
IT1    220
FR1    216
L1     198
Name: count, dtype: int64
Seasons: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [7]:
def attach_competition_meta(df_team_perf: pd.DataFrame, df_comps: pd.DataFrame) -> pd.DataFrame:
    """Attach readable competition metadata."""
    comps = df_comps[["competition_id", "competition_code", "name", "country_name"]].copy()
    comps["competition_id"] = comps["competition_id"].astype(str)

    perf = df_team_perf.copy()
    perf["competition_id"] = perf["competition_id"].astype(str)

    return perf.merge(comps, on="competition_id", how="left")


df_team_perf = attach_competition_meta(df_team_perf, df_comps)
print(df_team_perf.shape)
print(df_team_perf[["competition_id", "competition_code"]].drop_duplicates().sort_values("competition_id"))

(1074, 16)
   competition_id competition_code
0             ES1           laliga
20            FR1          ligue-1
40            GB1   premier-league
60            IT1          serie-a
80             L1       bundesliga


## Creating a Transfer spending dataset

In [8]:
def build_transfer_spending(df_transfers: pd.DataFrame) -> pd.DataFrame:
    """Build club-season transfer spending with season as start year."""

    transfers = df_transfers[
        ["to_club_id", "transfer_season", "transfer_fee"]
    ].copy()

    transfers = transfers.rename(columns={"to_club_id": "club_id"})

    # extract first year of season
    season_start = transfers["transfer_season"].astype(str).str.split("/").str[0]

    transfers["season"] = season_start.astype(int)
    transfers.loc[transfers["season"] < 100, "season"] += 2000

    # keep only seasons relevant for the project
    transfers = transfers[
        (transfers['season'] >= 2014) &
        (transfers['season'] <= 2024)
    ]

    # clean transfer fees
    transfers["transfer_fee"] = pd.to_numeric(
        transfers["transfer_fee"], errors="coerce"
    )

    # count nulls BEFORE dropping
    null_count = transfers["transfer_fee"].isna().sum()
    total_count = len(transfers)

    print(f"Removing {null_count} rows with missing transfer fees "
      f"({null_count / total_count:.2%} of data)")

    transfers = transfers.dropna(subset=["transfer_fee"])

    transfer_spending = (
        transfers.groupby(["club_id", "season"], as_index=False)
        .agg(transfer_spending=("transfer_fee", "sum"))
    )

    return transfer_spending

In [9]:
df_transfer_spending = build_transfer_spending(df_transfers)

df_transfer_spending["season"].min(), df_transfer_spending["season"].max()

Removing 26028 rows with missing transfer fees (33.23% of data)


(2014, 2024)

- Around 33% of transfer fees are missing, which is expected due to free or undisclosed transfers. Dropped since only total confirmed spending is measured and not total transfer activity

## Merging transfer spend with team performance

In [10]:
df_final = df_team_perf.merge(
    df_transfer_spending,
    on=["club_id", "season"],
    how="left"
)

df_final["transfer_spending"] = df_final["transfer_spending"].fillna(0)

In [11]:
df_final["transfer_spending"].describe()

count    1.074000e+03
mean     4.146458e+07
std      5.812689e+07
min      0.000000e+00
25%      5.250000e+06
50%      1.857500e+07
75%      5.466000e+07
max      5.743500e+08
Name: transfer_spending, dtype: float64

In [12]:
df_final.shape

(1074, 17)

In [13]:
df_final.head()

,season,competition_id,club_id,club_name,matches_played,wins,draws,losses,goals_for,goals_against,points,goal_difference,position,competition_code,name,country_name,transfer_spending
0,2014,ES1,131,Futbol Club Barcelona,38,30,4,4,110,21,94,89,1,laliga,laliga,Spain,42000000.0
1,2014,ES1,418,Real Madrid Club de Fútbol,38,30,2,6,118,38,92,80,2,laliga,laliga,Spain,35000000.0
2,2014,ES1,13,Club Atlético de Madrid S.A.D.,38,23,9,6,67,29,78,38,3,laliga,laliga,Spain,91350000.0
3,2014,ES1,1049,Valencia Club de Fútbol S. A. D.,38,22,11,5,70,32,77,38,4,laliga,laliga,Spain,18550000.0
4,2014,ES1,368,Sevilla Fútbol Club S.A.D.,38,23,7,8,71,45,76,26,5,laliga,laliga,Spain,3000000.0


In [14]:
df_final["transfer_spending"].min(), df_final["transfer_spending"].max()

(0.0, 574350000.0)

## Saving datasets

In [15]:
df_final.to_csv(
    "../data/intermediate/transf_perf.csv",
    index=False
)

df_team_perf.to_csv(
    "../data/intermediate/team_performance.csv",
    index=False
)

df_transfer_spending.to_csv(
    "../data/intermediate/transfer_spending.csv",
    index=False
)

In [16]:
df_final.head()

,season,competition_id,club_id,club_name,matches_played,wins,draws,losses,goals_for,goals_against,points,goal_difference,position,competition_code,name,country_name,transfer_spending
0,2014,ES1,131,Futbol Club Barcelona,38,30,4,4,110,21,94,89,1,laliga,laliga,Spain,42000000.0
1,2014,ES1,418,Real Madrid Club de Fútbol,38,30,2,6,118,38,92,80,2,laliga,laliga,Spain,35000000.0
2,2014,ES1,13,Club Atlético de Madrid S.A.D.,38,23,9,6,67,29,78,38,3,laliga,laliga,Spain,91350000.0
3,2014,ES1,1049,Valencia Club de Fútbol S. A. D.,38,22,11,5,70,32,77,38,4,laliga,laliga,Spain,18550000.0
4,2014,ES1,368,Sevilla Fútbol Club S.A.D.,38,23,7,8,71,45,76,26,5,laliga,laliga,Spain,3000000.0


In [17]:
df_team_perf.head()

,season,competition_id,club_id,club_name,matches_played,wins,draws,losses,goals_for,goals_against,points,goal_difference,position,competition_code,name,country_name
0,2014,ES1,131,Futbol Club Barcelona,38,30,4,4,110,21,94,89,1,laliga,laliga,Spain
1,2014,ES1,418,Real Madrid Club de Fútbol,38,30,2,6,118,38,92,80,2,laliga,laliga,Spain
2,2014,ES1,13,Club Atlético de Madrid S.A.D.,38,23,9,6,67,29,78,38,3,laliga,laliga,Spain
3,2014,ES1,1049,Valencia Club de Fútbol S. A. D.,38,22,11,5,70,32,77,38,4,laliga,laliga,Spain
4,2014,ES1,368,Sevilla Fútbol Club S.A.D.,38,23,7,8,71,45,76,26,5,laliga,laliga,Spain


In [18]:
df_transfer_spending.head()

,club_id,season,transfer_spending
0,1,2015,0.0
1,1,2017,0.0
2,1,2022,0.0
3,1,2024,0.0
4,2,2014,1000000.0


# Adding Squad Value for clubs

In [19]:
# Load datasets
tp = pd.read_csv('../data/intermediate/transf_perf.csv')
pv = pd.read_csv('../data/raw/squad value/player_valuations.csv')

# Prepare player valuations 
pv['date'] = pd.to_datetime(pv['date'])
pv['year'] = pv['date'].dt.year

# Filter to target leagues and seasons only
target_leagues = ['ES1', 'FR1', 'GB1', 'IT1', 'L1']
target_seasons = list(tp['season'].unique())
pv = pv[pv['player_club_domestic_competition_id'].isin(target_leagues)]
pv = pv[pv['year'].isin(target_seasons)]

# Aggregate to club-season level 
# For each club per season, take the max date valuation per player
# (most recent valuation within that year = end of season squad value)
pv_latest = (pv.sort_values('date')
               .groupby(['current_club_id', 'year'])
               .agg(squad_market_value=('market_value_in_eur', 'sum'),
                    num_players_valued=('player_id', 'nunique'))
               .reset_index())

pv_latest = pv_latest.rename(columns={
    'current_club_id': 'club_id',
    'year': 'season'
})

print(f"Squad valuations: {pv_latest.shape}")
print(pv_latest.head())

# Merge with transfer+performance dataset 
master = tp.merge(pv_latest, on=['club_id', 'season'], how='left')

print(f"\nMaster shape: {master.shape}")
print(f"Missing squad_market_value: {master['squad_market_value'].isna().sum()}")
print(f"\nSample:")
print(master[['club_name', 'season', 'competition_code', 'points', 
              'position', 'transfer_spending', 
              'squad_market_value']].head(10).to_string())

df_final = master.copy()

Squad valuations: (1915, 4)
   club_id  season  squad_market_value  num_players_valued
0        3    2014            89175000                  29
1        3    2015           126825000                  33
2        3    2016           188000000                  35
3        3    2017           329925000                  42
4        3    2018           138975000                  43

Master shape: (1074, 19)
Missing squad_market_value: 0

Sample:
                                          club_name  season competition_code  points  position  transfer_spending  squad_market_value
0                             Futbol Club Barcelona    2014           laliga      94         1         42000000.0           783100000
1                        Real Madrid Club de Fútbol    2014           laliga      92         2         35000000.0           861400000
2                    Club Atlético de Madrid S.A.D.    2014           laliga      78         3         91350000.0           576100000
3                

In [20]:
df_final["season"].min(), df_final["season"].max()

(2014, 2024)

In [21]:
# Check all unique club names per league
for league in df_final['competition_code'].unique():
    print(f"\n=== {league} ===")
    clubs = df_final[df_final['competition_code']==league]['club_name'].unique()
    for club in sorted(clubs):
        print(f"  {club}")


=== laliga ===
  Athletic Club Bilbao
  CD Leganés
  Club Atlético Osasuna
  Club Atlético de Madrid S.A.D.
  Cádiz CF
  Córdoba CF
  Deportivo Alavés S. A. D.
  Deportivo de La Coruña
  Elche Club de Fútbol S.A.D.
  Futbol Club Barcelona
  Getafe Club de Fútbol S. A. D. Team Dubai
  Girona Fútbol Club S. A. D.
  Granada CF
  Levante Unión Deportiva S.A.D.
  Málaga CF
  Rayo Vallecano de Madrid S. A. D.
  Real Betis Balompié S.A.D.
  Real Club Celta de Vigo S. A. D.
  Real Club Deportivo Mallorca S.A.D.
  Real Madrid Club de Fútbol
  Real Sociedad de Fútbol S.A.D.
  Real Valladolid CF
  Reial Club Deportiu Espanyol de Barcelona S.A.D.
  SD Eibar
  SD Huesca
  Sevilla Fútbol Club S.A.D.
  Sporting Gijón
  UD Almería
  UD Las Palmas
  Valencia Club de Fútbol S. A. D.
  Villarreal Club de Fútbol S.A.D.

=== ligue-1 ===
  AC Ajaccio
  AS Nancy-Lorraine
  AS Saint-Étienne
  Amiens SC
  Angers Sporting Club de l'Ouest
  Association de la Jeunesse auxerroise
  Association sportive de Monaco 

In [22]:
df_final['club_name'] = df_final['club_name'].str.replace(
    ' Team Dubai', '', regex=False
).str.strip()

In [23]:
df_final[df_final['club_name'].str.contains('Getafe', na=False)][['club_name','season']].drop_duplicates()

,club_name,season
13,Getafe Club de Fútbol S. A. D.,2014
116,Getafe Club de Fútbol S. A. D.,2015
301,Getafe Club de Fútbol S. A. D.,2017
397,Getafe Club de Fútbol S. A. D.,2018
497,Getafe Club de Fútbol S. A. D.,2019
602,Getafe Club de Fútbol S. A. D.,2020
700,Getafe Club de Fútbol S. A. D.,2021
798,Getafe Club de Fútbol S. A. D.,2022
893,Getafe Club de Fútbol S. A. D.,2023
995,Getafe Club de Fútbol S. A. D.,2024


# Renaming clubs for better readability

In [24]:
club_name_map = {
    # La Liga
    'Futbol Club Barcelona': 'FC Barcelona',
    'Real Madrid Club de Fútbol': 'Real Madrid FC',
    'Club Atlético de Madrid S.A.D.': 'Atlético Madrid',
    'Sevilla Fútbol Club S.A.D.': 'Sevilla FC',
    'Villarreal Club de Fútbol S.A.D.': 'Villarreal CF',
    'Real Betis Balompié S.A.D.': 'Real Betis',
    'Real Sociedad de Fútbol S.A.D.': 'Real Sociedad',
    'Valencia Club de Fútbol S. A. D.': 'Valencia CF',
    'Getafe Club de Fútbol S. A. D.': 'Getafe CF',
    'Rayo Vallecano de Madrid S. A. D.': 'Rayo Vallecano',
    'Real Club Celta de Vigo S. A. D.': 'Celta Vigo',
    'Real Club Deportivo Mallorca S.A.D.': 'RCD Mallorca',
    'Reial Club Deportiu Espanyol de Barcelona S.A.D.': 'RCD Espanyol',
    'Deportivo Alavés S. A. D.': 'Deportivo Alavés',
    'Levante Unión Deportiva S.A.D.': 'Levante UD',
    'Club Atlético Osasuna': 'Osasuna',
    'Elche Club de Fútbol S.A.D.': 'Elche CF',
    'Girona Fútbol Club S. A. D.': 'Girona FC',
    'Deportivo de La Coruña': 'Deportivo LC',

    # Ligue 1
    'Paris Saint-Germain Football Club': 'Paris Saint-Germain FC',
    'Olympique de Marseille': 'Marseille',
    'Olympique Lyonnais': 'Lyon',
    'Association sportive de Monaco Football Club': 'Monaco FC',
    'Lille Olympique Sporting Club': 'Lille',
    'Stade Rennais Football Club': 'Rennes',
    'Football Club de Nantes': 'Nantes',
    'Montpellier HSC': 'Montpellier',
    'Olympique Gymnaste Club Nice Côte d\'Azur': 'Nice',
    'Racing Club de Lens': 'Lens',
    'Racing Club de Strasbourg Alsace': 'Strasbourg',
    'Stade Reims': 'Reims',
    'Toulouse Football Club': 'Toulouse',
    'FC Girondins Bordeaux': 'Bordeaux',
    'Football Club Lorient-Bretagne Sud': 'Lorient',
    'AS Saint-Étienne': 'Saint-Étienne',
    'Stade brestois 29': 'Brest',
    'Clermont Foot 63': 'Clermont',
    'EA Guingamp': 'Guingamp',
    'SM Caen': 'Caen',
    'Football Club de Metz': 'Metz',
    'Association de la Jeunesse auxerroise': 'Auxerre',
    'ESTAC Troyes': 'Troyes',
    'Dijon FCO': 'Dijon',
    'Le Havre Athletic Club': 'Le Havre',
    'Nîmes Olympique': 'Nîmes',
    'Amiens SC': 'Amiens',
    'AS Nancy-Lorraine': 'Nancy',
    'SC Bastia': 'Bastia',
    'GFC Ajaccio': 'GFC Ajaccio',
    'AC Ajaccio': 'AC Ajaccio',
    'Thonon Évian Grand Genève FC': 'Évian TG',
    'Angers Sporting Club de l\'Ouest': 'Angers SCO',

    # Premier League
    'Arsenal Football Club': 'Arsenal',
    'Chelsea Football Club': 'Chelsea',
    'Liverpool Football Club': 'Liverpool',
    'Manchester City Football Club': 'Manchester City',
    'Manchester United Football Club': 'Manchester United',
    'Tottenham Hotspur Football Club': 'Tottenham',
    'Newcastle United Football Club': 'Newcastle United',
    'West Ham United Football Club': 'West Ham',
    'Aston Villa Football Club': 'Aston Villa',
    'Brighton and Hove Albion Football Club': 'Brighton',
    'Wolverhampton Wanderers Football Club': 'Wolves',
    'Brentford Football Club': 'Brentford',
    'Nottingham Forest Football Club': 'Nottingham Forest',
    'Crystal Palace Football Club': 'Crystal Palace',
    'Everton Football Club': 'Everton',
    'Fulham Football Club': 'Fulham',
    'Burnley Football Club': 'Burnley',
    'Association Football Club Bournemouth': 'Bournemouth',
    'Sunderland Association Football Club': 'Sunderland',
    'Leeds United Association Football Club': 'Leeds United',
    'Southampton FC': 'Southampton FC',
    'Watford FC': 'Watford FC',
    'West Bromwich Albion': 'West Brom',
    'Queens Park Rangers': 'Queens Park Rangers',

    # Serie A
    'Juventus Football Club': 'Juventus',
    'Football Club Internazionale Milano S.p.A.': 'Inter Milan',
    'Associazione Calcio Milan': 'AC Milan',
    'Associazione Sportiva Roma': 'AS Roma',
    'Società Sportiva Lazio S.p.A.': 'Lazio',
    'Società Sportiva Calcio Napoli': 'Napoli',
    'Atalanta Bergamasca Calcio S.p.a.': 'Atalanta',
    'Associazione Calcio Fiorentina': 'Fiorentina',
    'Bologna Football Club 1909': 'Bologna',
    'Udinese Calcio': 'Udinese',
    'Torino Calcio': 'Torino',
    'Genoa Cricket and Football Club': 'Genoa',
    'UC Sampdoria': 'Sampdoria',
    'Cagliari Calcio': 'Cagliari',
    'FC Empoli': 'Empoli',
    'Unione Sportiva Sassuolo Calcio': 'Sassuolo',
    'Unione Sportiva Lecce': 'Lecce',
    'Unione Sportiva Cremonese S.p.A.': 'Cremonese',
    'Verona Hellas Football Club': 'Hellas Verona',
    'Spezia Calcio': 'Spezia',
    'AC Monza': 'Monza',
    'Benevento Calcio': 'Benevento',
    'Frosinone Calcio': 'Frosinone',
    'US Salernitana 1919': 'Salernitana',
    'Brescia Calcio': 'Brescia',
    'SPAL': 'SPAL',
    'Palermo FC': 'Palermo',
    'Delfino Pescara 1936': 'Pescara',
    'AC Carpi': 'Carpi',
    'Cesena FC': 'Cesena',
    'Parma Calcio 1913': 'Parma',
    'Venezia FC': 'Venezia',
    'Calcio Como': 'Como',

    # Bundesliga
    'FC Bayern München': 'Bayern Munich',
    'Borussia Dortmund': 'Borussia Dortmund BVB',
    'Bayer 04 Leverkusen Fußball': 'Bayer Leverkusen',
    'RasenBallsport Leipzig': 'RB Leipzig',
    'Borussia Verein für Leibesübungen 1900 Mönchengladbach': 'Borussia Mönchengladbach',
    'Eintracht Frankfurt Fußball AG': 'Eintracht Frankfurt',
    'Sport-Club Freiburg': 'SC Freiburg',
    'Turn- und Sportgemeinschaft 1899 Hoffenheim Fußball-Spielbetriebs': 'Hoffenheim',
    'Verein für Bewegungsspiele Stuttgart 1893': 'VfB Stuttgart',
    'Verein für Leibesübungen Wolfsburg': 'VfL Wolfsburg',
    'Sportverein Werder Bremen von 1899': 'Werder Bremen',
    'FC Schalke 04': 'Schalke 04',
    'Hamburger Sport Verein': 'Hamburger SV',
    '1. Fußball- und Sportverein Mainz 05': 'Mainz 05',
    '1. Fußball-Club Köln': 'FC Köln',
    '1. Fußballclub Union Berlin': 'Union Berlin',
    '1. Fußballclub Heidenheim 1846': 'Heidenheim',
    'Hertha BSC': 'Hertha BSC',
    'Arminia Bielefeld': 'Arminia Bielefeld',
    'Hannover 96': 'Hannover 96',
    'Fußball-Club Augsburg 1907': 'FC Augsburg',
    'Fußball-Club St. Pauli von 1910': 'St. Pauli',
    'Holstein Kiel': 'Holstein Kiel',
    'SV Darmstadt 98': 'Darmstadt 98',
    'SC Paderborn 07': 'Paderborn',
    'SpVgg Greuther Fürth': 'Greuther Fürth',
    'FC Ingolstadt 04': 'FC Ingolstadt',
    '1.FC Nuremberg': 'Nürnberg',
    'Fortuna Düsseldorf': 'Fortuna Düsseldorf',
}

df_final['club_name'] = df_final['club_name'].replace(club_name_map)

# Verify — check any names NOT in the map (shouldn't be any)
all_clubs = set(df_final['club_name'].unique())
mapped = set(club_name_map.values())
print(f"Total unique club names after rename: {len(all_clubs)}")
print(f"Any long names remaining:")
long_names = [c for c in all_clubs if len(c) > 25]
for c in sorted(long_names):
    print(f"  {c}")

Total unique club names after rename: 164
Any long names remaining:


# Checking for redundancy

In [25]:
df_final["competition_code"].value_counts()

competition_code
laliga            220
premier-league    220
serie-a           220
ligue-1           216
bundesliga        198
Name: count, dtype: int64

In [26]:
df_final["name"].value_counts()

name
laliga            220
premier-league    220
serie-a           220
ligue-1           216
bundesliga        198
Name: count, dtype: int64

In [27]:
df_final = df_final.drop(columns=['competition_code'])
df_final = df_final.rename(columns={'name': 'league'})
print(df_final[['competition_id', 'league', 'country_name']].drop_duplicates())

   competition_id          league country_name
0             ES1          laliga        Spain
20            FR1         ligue-1       France
40            GB1  premier-league      England
60            IT1         serie-a        Italy
80             L1      bundesliga      Germany


In [28]:
# Check how many matches per club in 2024
games_2024 = df_final[df_final['season'] == 2024]
print(games_2024.groupby(['league','club_name'])['matches_played'].max().unstack('league').to_string())

league                    bundesliga  laliga  ligue-1  premier-league  serie-a
club_name                                                                     
AC Milan                         NaN     NaN      NaN             NaN     19.0
AS Roma                          NaN     NaN      NaN             NaN     16.0
Angers SCO                       NaN     NaN     34.0             NaN      NaN
Arsenal                          NaN     NaN      NaN            28.0      NaN
Aston Villa                      NaN     NaN      NaN            24.0      NaN
Atalanta                         NaN     NaN      NaN             NaN     15.0
Athletic Club Bilbao             NaN    38.0      NaN             NaN      NaN
Atlético Madrid                  NaN    38.0      NaN             NaN      NaN
Auxerre                          NaN     NaN     34.0             NaN      NaN
Bayer Leverkusen                19.0     NaN      NaN             NaN      NaN
Bayern Munich                   20.0     NaN      Na

- 2024 league standings are incomplete, so drop 2024.

In [29]:
# Drop 2024 and rename/add season column
df_final = df_final[df_final['season'] != 2024].copy()

# Rename season to year
df_final = df_final.rename(columns={'season': 'year'})

# Add season column
df_final['season'] = df_final['year'].astype(str) + '/' + (df_final['year'] + 1).astype(str).str[-2:]

# Verify
print(f"Shape: {df_final.shape}")
print(df_final[['year', 'season']].drop_duplicates().sort_values('year').to_string())

Shape: (978, 19)
     year   season
0    2014  2014/15
98   2015  2015/16
196  2016  2016/17
294  2017  2017/18
392  2018  2018/19
490  2019  2019/20
588  2020  2020/21
686  2021  2021/22
784  2022  2022/23
882  2023  2023/24


# Adding 'Clean Sheets' to performance data

In [30]:
TOP5_IDS = ['GB1', 'ES1', 'IT1', 'FR1', 'L1']
df_games = df_games[
    (df_games['competition_id'].isin(TOP5_IDS)) &
    (df_games['season'] >= 2014) &
    (df_games['season'] <= 2023)
]

# Home clean sheets (away goals == 0)
home_cs = df_games[df_games['away_club_goals'] == 0].groupby(
    ['home_club_id', 'season']
).size().reset_index(name='home_clean_sheets')
home_cs = home_cs.rename(columns={'home_club_id': 'club_id'})

# Away clean sheets (home goals == 0)
away_cs = df_games[df_games['home_club_goals'] == 0].groupby(
    ['away_club_id', 'season']
).size().reset_index(name='away_clean_sheets')
away_cs = away_cs.rename(columns={'away_club_id': 'club_id'})

# Combine
clean_sheets = home_cs.merge(away_cs, on=['club_id', 'season'], how='outer').fillna(0)
clean_sheets['clean_sheets'] = (
    clean_sheets['home_clean_sheets'] + clean_sheets['away_clean_sheets']
).astype(int)
clean_sheets = clean_sheets[['club_id', 'season', 'clean_sheets']]
clean_sheets = clean_sheets.rename(columns={'season': 'year'})

# Merge into df_final
df_final = df_final.merge(clean_sheets, on=['club_id', 'year'], how='left')
print(f"✓ Clean sheets added — missing: {df_final['clean_sheets'].isna().sum()}")

✓ Clean sheets added — missing: 0


# Saving the final dataset

In [31]:
os.makedirs('../data/final', exist_ok=True)
df_final.to_csv('../data/final/master_football_dataset.csv', index=False)

# sanity check
print(f"Saved: {df_final.shape}")
print(f"Seasons: {df_final['season'].min()} – {df_final['season'].max()}")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(df_final.columns)

Saved: (978, 20)
Seasons: 2014/15 – 2023/24
Missing values: 0
Index(['year', 'competition_id', 'club_id', 'club_name', 'matches_played',
       'wins', 'draws', 'losses', 'goals_for', 'goals_against', 'points',
       'goal_difference', 'position', 'league', 'country_name',
       'transfer_spending', 'squad_market_value', 'num_players_valued',
       'season', 'clean_sheets'],
      dtype='object')
